# SignBridge AI: GRU Sequence Classifier Training in Google Colab
### Step 3: Deep Learning Baseline for Indian Sign Language (ISL) Sentence Recognition

This notebook trains the first **closed-set sentence classification model** for SignBridge AI on Google Colab (with optional GPU acceleration).

#### Key Highlights:
- **Feature Topology:** Candidate B 3D (229 dimensions per frame: 99 pose + 63 left hand + 63 right hand + 4 indicators).
- **Model:** Bidirectional Gated Recurrent Unit (Bi-GRU) with variable-length sequence handling.
- **Signer Isolation:** Training on Signers 1–5, Validation on Signer 6, and Test on Signer 7.
- **Zero Data Fabrication:** Missing hand landmarks are explicitly zero-filled without linear interpolation.

## 1. Setup & Environment Verification

In [ ]:
# Clone repository or ensure you are in the workspace root
!git status

import torch
import numpy as np
import json
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU.")

## 2. Ingest Preprocessed Tensors
Mount your Google Drive or ensure `data/processed/isl_csltr_tensors_v2/` is present in the workspace.

In [ ]:
data_dir = Path("data/processed/isl_csltr_tensors_v2")
for split in ["train", "validation", "test", "quarantine"]:
    p = data_dir / split
    count = len(list(p.glob("*.npz"))) if p.exists() else 0
    print(f"{split:12s}: {count} tensors found")

## 3. Dataset Pre-Flight Validation

In [ ]:
!python scripts/validate_ml_dataset.py --data-dir data/processed/isl_csltr_tensors_v2

## 4. Train the GRU Sequence Classifier

In [ ]:
!python scripts/train_gru.py \
    --epochs 100 \
    --batch-size 16 \
    --lr 0.001 \
    --hidden-dim 128 \
    --num-layers 1 \
    --dropout 0.2 \
    --patience 10 \
    --data-dir data/processed/isl_csltr_tensors_v2 \
    --checkpoint-dir models/checkpoints

## 5. Evaluate on Held-Out Test Set (Signer 7)

In [ ]:
!python scripts/evaluate_gru.py \
    --checkpoint models/checkpoints/best_gru_model.pt \
    --data-dir data/processed/isl_csltr_tensors_v2

## 6. Interactive Inference Example

In [ ]:
from backend.app.ml.inference import SignBridgeInferenceEngine

engine = SignBridgeInferenceEngine.load_from_checkpoint("models/checkpoints/best_gru_model.pt")

# Load a sample test tensor
test_files = list(Path("data/processed/isl_csltr_tensors_v2/test").glob("*.npz"))
if test_files:
    sample_path = test_files[0]
    with np.load(sample_path, allow_pickle=True) as data:
        feat = data["features"]
        meta = json.loads(str(data["metadata"]))
        
    result = engine.predict(feat, confidence_threshold=0.50)
    print("--- Sample Ground Truth ---")
    print(f"Sentence: {meta.get('sentence_text')}")
    print("--- Model Prediction ---")
    print(f"Predicted Label: {result['predicted_label']}")
    print(f"Confidence: {result['confidence']:.4f}")
    print(f"Status: {result['status']}")
    print(f"Top-3 Predictions: {result['top_k'][:3]}")